In [1]:
# Importation des bibliothèques
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from ucimlrepo import fetch_ucirepo 

In [ ]:
csv_path = "../PhiUSIIL_Phishing_URL_Dataset.csv"
try:
    df = pd.read_csv(csv_path)
    print("Dataset chargé avec succès !")
except Exception as e:
    print(f"Erreur lors du chargement du dataset : {e}")

print(df.columns)


# Sélection des caractéristiques souhaitées
features_to_use = [
    'URLLength', 'DomainLength', 'NoOfSubDomain', 'IsDomainIP',
    'NoOfLettersInURL', 'NoOfDegitsInURL', 'NoOfEqualsInURL',
    'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL',
    'SpacialCharRatioInURL', 'TLDLength'
]

# On crée le DataFrame final avec les caractéristiques (X) et la cible (y)
X = df[features_to_use]
Y = df['label']

print("\nCaractéristiques sélectionnées :")
print(X.head())
print(Y.head())
print(f"\nNombre d'échantillons: {len(df)}")

In [ ]:
# Initialisation du normalisateur
scaler = MinMaxScaler()

# Normalisation des caractéristiques
X_scaled = scaler.fit_transform(X)

# Affichage des 5 premières lignes normalisées pour vérifier
print("Données après normalisation (5 premières lignes) :")
print(X_scaled[:5])

In [ ]:
# Division des données en 80% pour l'entraînement et 20% pour la validation
X_train, X_val, Y_train, Y_val = train_test_split(
    X_scaled, 
    Y, 
    test_size=0.2, 
    random_state=42, # Pour la reproductibilité des résultats
    stratify=Y # Assure que la proportion de labels est la même dans les deux ensembles
)

print(f"Taille de l'ensemble d'entraînement : {X_train.shape[0]} échantillons")
print(f"Taille de l'ensemble de validation : {X_val.shape[0]} échantillons")

In [ ]:
# Définition du modèle séquentiel
model = Sequential()

# Nombre de caractéristiques en entrée
input_dim = X_train.shape[1]

model.add(Dense(128, input_dim=input_dim, activation='relu'))

model.add(Dense(64, activation='relu'))

model.add(Dense(32, activation='relu'))

model.add(Dense(1, activation='sigmoid'))


# Affichage d'un résumé de l'architecture du modèle
model.summary()

In [ ]:
# Compilation du modèle
model.compile(
    optimizer='adam',
    loss='binary_crossentropy', # fonction de coût pour la classification binaire
    metrics=['accuracy']
)

In [ ]:
# Importer le callback
from tensorflow.keras.callbacks import EarlyStopping

early_stop = False
EPOCHS = 50
BATCH_SIZE = 64
print("Début de l'entraînement...")

if early_stop:
    # 'patience=5' : on attend 5 époques après la dernière amélioration avant d'arrêter.
    early_stopper = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    EPOCHS = 100

    history = model.fit(
    X_train,
    Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, Y_val),
    callbacks=[early_stopper],
    verbose=1
    )
else:
    history = model.fit(
    X_train,
    Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, Y_val),
    verbose=1
    )
    
print("\nEntraînement terminé !")

In [ ]:
import matplotlib.pyplot as plt

# Fonction pour afficher les courbes de perte et de précision
def plot_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Courbe de précision
    ax1.plot(history.history['accuracy'], label='Données d\'entraînement')
    ax1.plot(history.history['val_accuracy'], label='Données de validation')
    ax1.set_title('Précision du modèle')
    ax1.set_xlabel('Époque')
    ax1.set_ylabel('Précision (Accuracy)')
    ax1.legend(loc='lower right')
    ax1.grid()
    
    # Courbe de perte
    ax2.plot(history.history['loss'], label='Données d\'entraînement')
    ax2.plot(history.history['val_loss'], label='Données de validation')
    ax2.set_title('Perte du modèle')
    ax2.set_xlabel('Époque')
    ax2.set_ylabel('Perte (Loss)')
    ax2.legend(loc='upper right')
    ax2.grid()
    
    plt.show()

plot_history(history)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns

loss, accuracy = model.evaluate(X_val, Y_val, verbose=0)
print(f"--- Évaluation finale sur l'ensemble de validation ---")
print(f"Perte (Loss)     : {loss:.4f}")
print(f"Précision (Accuracy) : {accuracy * 100:.2f}%\n")


y_pred_probs = model.predict(X_val)
y_pred_classes = (y_pred_probs > 0.5).astype(int)



print("--- Rapport de Classification ---")
target_names = ['Légitime (0)', 'Phishing (1)'] 
print(classification_report(Y_val, y_pred_classes, target_names=target_names))

print("--- Matrice de Confusion ---")
cm = confusion_matrix(Y_val, y_pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Prédit Légitime', 'Prédit Phishing'], 
            yticklabels=['Réel Légitime', 'Réel Phishing'])
plt.title('Matrice de Confusion')
plt.show()